# Flagship 1 — Config-driven historical shock profiles

Read `config/event_windows.csv` for all 7 windows. The 2025–26 tariff cases are **provisional** and not training eligible by default. Report only observed industrial production YoY changes (not output recovery levels or causal impacts).

In [1]:
from pathlib import Path
import sys
for loc in [Path.cwd(),Path.cwd()/'notebooks',Path.cwd()/'research_expansion'/'notebooks']:
    if (loc/'common.py').exists():
        sys.path.insert(0,str(loc.resolve()));break
from common import root, load_panel, load_cases, output

In [2]:
import pandas as pd, numpy as np
from common import event_config
p = load_panel()
cfg = event_config()
rows = []
coverage = []
excluded = []
for (country,industry), grp in p.groupby(['country','industry_code']):
    grp = grp.sort_values('date')
    months = grp['date'].dt.to_period('M').astype(str)
    for _,r in cfg.iterrows():
        def months_in(lo,hi):
            if not lo or not hi:
                return pd.Series([], dtype=float)
            mask = (months>=lo)&(months<=hi)
            return grp.loc[mask,'production_yoy_pct'].dropna()
        b = months_in(r.baseline_start,r.baseline_end)
        s = months_in(r.shock_start,r.shock_end)
        f = months_in(r.post_start,r.post_end)
        expected = [len(pd.period_range(lo, hi, freq='M')) if lo and hi else 0
                    for lo, hi in [(r.baseline_start,r.baseline_end), (r.shock_start,r.shock_end), (r.post_start,r.post_end)]]
        counts = [len(b), len(s), len(f)]
        status = ('ready' if counts == expected else 'partial') if min(counts[:2]) >= 6 else ('partial' if len(s) else 'unavailable')
        coverage.append(dict(country=country,industry_code=industry,event=r.event,status=status,
                             eligible=min(counts[:2])>=6,n_baseline=len(b),n_shock=len(s),n_post=len(f),
                             expected_baseline=expected[0],expected_shock=expected[1],expected_post=expected[2]))
        if len(b)<6 or len(s)<6:
            excluded.append({'country':country,'industry_code':industry,'event':r.event,'n_baseline':len(b),'n_shock':len(s),'reason':'fewer than 6 observed monthly values in baseline or shock'})
            continue
        post = grp.loc[(months>=r.post_start)&(months<=r.post_end),'production_yoy_pct'] if r.post_start and len(f)>=6 else pd.Series(dtype=float)
        sustained = (post.ge(b.mean()) & post.notna()).rolling(3).sum().eq(3)
        normalized_month = grp.loc[sustained[sustained].index[0],'date'].strftime('%Y-%m') if sustained.any() else None
        rows.append(dict(country=country,industry_code=industry,event=r.event,
          event_status=r.get('status',''),event_category=r.get('category',''),
          baseline_mean_yoy=b.mean(),baseline_std_yoy=b.std(ddof=0),
          baseline_negative_share=(b<0).mean(),shock_mean_yoy=s.mean(),
          delta_pp=s.mean()-b.mean(),worst_yoy=s.min(),negative_share=(s<0).mean(),
          shock_std_yoy=s.std(ddof=0),post_mean_yoy=f.mean() if len(f) else np.nan,
          growth_normalization_month=normalized_month,
          post_shift_pp=f.mean()-s.mean() if len(f)>=6 else np.nan,
          weakest_3mo_growth_pct=grp.loc[(months>=r.shock_start)&(months<=r.shock_end),'production_yoy_pct'].rolling(3).mean().min(),
          n_baseline=len(b),n_shock=len(s),n_post=len(f),
          baseline_window=f'{r.baseline_start}/{r.baseline_end}',
          shock_window=f'{r.shock_start}/{r.shock_end}',
          post_window=f'{r.post_start}/{r.post_end}' if r.post_start else '',
          post_interpretation=('observed_followup_only' if len(f)>=6 else 'insufficient_for_followup')))
cases=pd.DataFrame(rows)
if cases.empty:
    raise ValueError('No observed country-industry-events with >=6 monthly baseline AND shock observations')
cases['rank'] = cases.groupby(['country','event'])['delta_pp'].rank(method='min',ascending=False).astype(int)
cases['rank_count'] = cases.groupby(['country','event'])['delta_pp'].transform('size')
# Rank only the observed cases meeting the six-month gate.
coverage_path = root()/'reports'/'research'/'event_coverage.json'
from common import json
coverage_path.write_text(json.dumps(coverage, separators=(',', ':')), encoding='utf-8')
assert len(coverage) == len(cfg) * 30
path=root()/'data'/'processed'/'case_features.csv'
path.parent.mkdir(parents=True,exist_ok=True)
cases.to_csv(path,index=False)
display(cases.sort_values('delta_pp').head(15))
output('shock_profiles',{'model':'adapted_growth_resistance','status':'fitted',
  'n_target_events':len(cfg),'target_case_grid':len(cfg)*30,
  'cases':len(cases),'n_countries':cases.country.nunique(),'n_industries':cases.industry_code.nunique(),
  'metric':'shock minus baseline mean YoY production growth in percentage points',
  'limitations':['Descriptive association only; overlapping event windows and tariff policy changes',
                 'India YoY rates cannot establish recovery of production level',
                 '2025/26 tariffs provisional with no confirmed recovery; some countries lack 2026 values',
                 'COVID 2021 YoY rebound includes denominator base effects'],
  'excluded_cases':excluded,
  'rows':cases.astype(object).where(pd.notna(cases),None).to_dict('records')})


,country,industry_code,event,event_status,event_category,baseline_mean_yoy,baseline_std_yoy,baseline_negative_share,shock_mean_yoy,delta_pp,...,weakest_3mo_growth_pct,n_baseline,n_shock,n_post,baseline_window,shock_window,post_window,post_interpretation,rank,rank_count
126,IND,29,covid_disruption_2020,core,pandemic,0.584167,16.494059,0.583333,-29.387000,-29.971167,...,-77.006667,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,10,10
78,IND,13,covid_disruption_2020,core,pandemic,0.050417,3.961001,0.458333,-27.783000,-27.833417,...,-66.243333,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,9,10
114,IND,27,covid_disruption_2020,core,pandemic,0.311667,6.625481,0.541667,-22.747000,-23.058667,...,-67.183333,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,8,10
120,IND,28,covid_disruption_2020,core,pandemic,-1.353333,12.120385,0.625000,-23.355000,-22.001667,...,-63.533333,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,7,10
102,IND,24,covid_disruption_2020,core,pandemic,8.046667,6.619549,0.041667,-12.604000,-20.650667,...,-42.920000,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,6,10
89,IND,21,us_china_tariffs_2018,core,trade_policy,21.812333,15.255194,0.100000,2.707778,-19.104556,...,-0.993333,30,18,2,2016-01/2018-06,2018-07/2019-12,2020-01/2020-02,insufficient_for_followup,10,10
108,IND,25,covid_disruption_2020,core,pandemic,-3.881250,11.638525,0.625000,-21.790000,-17.908750,...,-62.030000,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,5,10
195,USA,29,covid_disruption_2020,core,pandemic,1.084508,5.206428,0.333333,-16.542315,-17.626823,...,-55.884482,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,10,10
65,DEU,29,covid_disruption_2020,core,pandemic,-3.128570,8.279629,0.625000,-20.724770,-17.596199,...,-56.782783,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,10,10
72,IND,10,covid_disruption_2020,core,pandemic,11.305000,8.498506,0.125000,-6.248000,-17.553000,...,-18.133333,24,10,12,2018-01/2019-12,2020-03/2020-12,2021-01/2021-12,observed_followup_only,4,10


Saved C:\Users\mehak\OneDrive\ドキュメント\ChatGPT\Industry Res\reports\research\models\shock_profiles.json


WindowsPath('C:/Users/mehak/OneDrive/ドキュメント/ChatGPT/Industry Res/reports/research/models/shock_profiles.json')